# 037 — Previsões operacionais de admissões (produto)

Gera o lote operacional de previsões mensais de admissões para as 698 séries UF × subgrupo CBO a partir do modelo registrado `workspace.gold.demanda_admissoes_lgbm` (v1), gravando em `workspace.gold.previsoes_admissoes`. **Método recursivo**: cada mês é previsto como h=1 (o horizonte validado no backtest do 036) e a previsão realimenta o histórico para o mês seguinte. O histórico vem da wide table atualizada até o último mês real disponível — substituindo o histórico embutido no artefato, que congela no treino.

In [0]:
%pip install lightgbm
dbutils.library.restartPython()

In [0]:
# Configurações, parâmetros e carga do modelo registrado
import mlflow
import pandas as pd
import numpy as np
from datetime import datetime
from pyspark.sql import functions as F

dbutils.widgets.text("tabela_features", "workspace.gold.ml_features_mensal_uf_cbo", "Wide table de features (fonte do histórico)")
dbutils.widgets.text("versao_transformacao", "0.1.0", "Versão da wide table a consumir")
dbutils.widgets.text("tabela_previsoes", "workspace.gold.previsoes_admissoes", "Tabela-alvo das previsões")
dbutils.widgets.text("versao_previsao", "0.1.0", "Versão deste lote de previsões")
dbutils.widgets.text("horizonte", "12", "Meses à frente (recursivo)")
dbutils.widgets.text("model_uri", "models:/workspace.gold.demanda_admissoes_lgbm/1", "Modelo registrado no UC")

TABELA_FEATURES = dbutils.widgets.get("tabela_features")
VERSAO = dbutils.widgets.get("versao_transformacao")
TABELA_PREVISOES = dbutils.widgets.get("tabela_previsoes")
VERSAO_PREVISAO = dbutils.widgets.get("versao_previsao")
HORIZONTE = int(dbutils.widgets.get("horizonte"))
MODEL_URI = dbutils.widgets.get("model_uri")

SILVER_CAGED = "workspace.silver.novo_caged_movimentacoes"
DT_PROCESSAMENTO = datetime.now()

# De-para fixo e documentado: código IBGE (CAGED) → sigla (mesmo do 035)
UF_SIGLA = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL",
    28: "SE", 29: "BA", 31: "MG", 32: "ES", 33: "RJ", 35: "SP", 41: "PR",
    42: "SC", 43: "RS", 50: "MS", 51: "MT", 52: "GO", 53: "DF",
}

pyfunc = mlflow.pyfunc.load_model(MODEL_URI)
modelo = pyfunc.unwrap_python_model()
print(f"Modelo carregado: {MODEL_URI}")
print(f"Wrapper: {type(modelo).__name__} | features: {len(modelo.features)} | "
      f"último mês do histórico EMBUTIDO: {modelo.hist['competencia_mov'].max():%Y-%m} (será substituído abaixo)")

In [0]:
# Histórico ATUALIZADO da wide table — substitui o histórico embutido no artefato (congelado no treino).
# O modelo final NÃO usa features INEP (descartadas no screening do 035), então a wide table
# é fonte completa de y/desl/salário até o último mês real disponível.
KEYS = ["sigla_uf", "codigo_cbo_subgrupo"]

df_wide = spark.table(TABELA_FEATURES).where(F.col("versao_transformacao") == VERSAO)
ultimo_mes_real = df_wide.agg(F.max("competencia_mov")).first()[0]
mes_base = pd.Timestamp(ultimo_mes_real)
inicio_hist = mes_base - pd.DateOffset(months=12)  # 13 meses: lags 1..12 completos para o 1º horizonte

hist = (
    df_wide.where(F.col("competencia_mov") >= F.lit(inicio_hist.date()))
    .select("sigla_uf", "codigo_cbo_subgrupo", "competencia_mov", "admissoes", "desligamentos", "salario_mediano_lag_1")
    .orderBy("sigla_uf", "codigo_cbo_subgrupo", "competencia_mov")
    .toPandas()
)
hist["competencia_mov"] = pd.to_datetime(hist["competencia_mov"])
# sal(m) = salário mediano do mês m = salario_mediano_lag_1 da linha m+1 (mesma construção do wrapper no 036)
hist["sal"] = hist.groupby(KEYS, observed=True)["salario_mediano_lag_1"].shift(-1)

# Salário do ÚLTIMO mês real (não existe linha m+1 na wide) — direto da silver, um mês só
subgrupos = sorted(hist["codigo_cbo_subgrupo"].unique())
df_sal = (
    spark.table(SILVER_CAGED)
    .where((F.col("competencia_mov") == F.lit(ultimo_mes_real))
           & (F.col("uf") != 99) & (F.col("saldo_movimentacao") == 1))
    .withColumn("codigo_cbo_subgrupo", F.substring(F.lpad(F.col("cbo2002_ocupacao").cast("string"), 6, "0"), 1, 3))
    .where(F.col("codigo_cbo_subgrupo").isin(subgrupos))
    .groupBy("uf", "codigo_cbo_subgrupo")
    .agg(F.percentile_approx("salario", 0.5).alias("sal_ultimo"))
    .toPandas()
)
df_sal["sigla_uf"] = df_sal["uf"].map(UF_SIGLA)
df_sal["sal_ultimo"] = pd.to_numeric(df_sal["sal_ultimo"], errors="coerce").astype("float64")  # silver retorna Decimal
map_sal = {(r.sigla_uf, r.codigo_cbo_subgrupo): r.sal_ultimo for r in df_sal.itertuples()}
mascara_ultimo = hist["competencia_mov"] == mes_base
hist.loc[mascara_ultimo, "sal"] = [map_sal.get((u, s)) for u, s in
                                  zip(hist.loc[mascara_ultimo, "sigla_uf"], hist.loc[mascara_ultimo, "codigo_cbo_subgrupo"])]
hist["sal"] = pd.to_numeric(hist["sal"], errors="coerce").astype("float64")  # garante float64 (não object)

# Interface do wrapper: chaves como CATEGORIAS DO TREINO (cat_maps do próprio modelo)
for c, cats in modelo.cat_maps.items():
    hist[c] = pd.Categorical(hist[c], categories=cats)
modelo.hist = hist.rename(columns={"admissoes": "y", "desligamentos": "desl"})[KEYS + ["competencia_mov", "y", "desl", "sal"]]

qtd_series = hist.groupby(KEYS, observed=True).ngroups
assert qtd_series == 698, f"Séries inesperadas no histórico: {qtd_series}"
assert hist["admissoes"].notna().all(), "Alvo nulo no histórico"
print(f"Histórico reconstruído: {qtd_series} séries, {hist['competencia_mov'].min():%Y-%m} a {hist['competencia_mov'].max():%Y-%m}")
print(f"Salário do último mês real ({mes_base:%Y-%m}): preenchido para {hist.loc[mascara_ultimo, 'sal'].notna().sum()} de {int(mascara_ultimo.sum())} séries")

In [0]:
# Previsão RECURSIVA h=1..H — cada mês previsto como h=1 (horizonte VALIDADO no backtest),
# com realimentação de y para o mês seguinte. Desligamentos e salário futuros são desconhecidos
# → NaN (LightGBM trata nativamente; limitação documentada nas observações).
series_str = hist[KEYS].drop_duplicates().astype(str).reset_index(drop=True)
registros = []
for h in range(1, HORIZONTE + 1):
    mes_prev = mes_base + pd.DateOffset(months=h)
    fut = series_str.copy()
    fut["competencia_mov"] = mes_prev
    saida = pyfunc.predict(fut)  # linhas futuras → coluna forecast
    assert saida["forecast"].notna().all(), f"Previsões nulas no horizonte {h}"
    saida["horizonte_h"] = h
    registros.append(saida)

    # Realimenta o histórico (recursivo) — chaves reconvertidas às categorias do treino
    alimentar = saida[KEYS + ["competencia_mov"]].copy()
    alimentar["y"] = saida["forecast"].to_numpy()
    alimentar["desl"] = np.nan
    alimentar["sal"] = np.nan
    for c, cats in modelo.cat_maps.items():
        alimentar[c] = pd.Categorical(alimentar[c].astype(str), categories=cats)
    modelo.hist = pd.concat([modelo.hist, alimentar[KEYS + ["competencia_mov", "y", "desl", "sal"]]], ignore_index=True)

df_prev = pd.concat(registros, ignore_index=True)
meses_previstos = df_prev["competencia_mov"].nunique()
print(f"Previsões geradas: {len(df_prev)} linhas = {qtd_series} séries × {meses_previstos} meses "
      f"({df_prev['competencia_mov'].min():%Y-%m} a {df_prev['competencia_mov'].max():%Y-%m})")

# Sanidade do h=1: total previsto do próximo mês vs. total real do último mês (mesma massa de séries)
total_h1 = df_prev.loc[df_prev["horizonte_h"] == 1, "forecast"].sum()
total_ultimo_real = hist.loc[hist["competencia_mov"] == mes_base, "admissoes"].sum()
print(f"h=1 ({mes_base + pd.DateOffset(months=1):%Y-%m}): total previsto {total_h1:,.0f} admissões "
      f"vs. último mês real {mes_base:%Y-%m} = {total_ultimo_real:,.0f} (razão {total_h1 / total_ultimo_real:.2f})")

In [0]:
# Gravação idempotente na gold — append por versao_previsao (skip se a versão já existe)
negativos = int((df_prev["forecast"] < 0).sum())
df_prev["previsao_admissoes"] = df_prev["forecast"].clip(lower=0).round(1)  # admissões não podem ser negativas
df_prev["previsao_bruta"] = df_prev["forecast"]
df_prev = df_prev.rename(columns={"competencia_mov": "competencia_prevista"})
df_prev["modelo"] = "demanda_admissoes_lgbm"
df_prev["versao_modelo"] = MODEL_URI.split("/")[-1]
df_prev["versao_features"] = VERSAO
df_prev["ultimo_mes_real"] = mes_base
df_prev["versao_previsao"] = VERSAO_PREVISAO
df_prev["dt_processamento"] = DT_PROCESSAMENTO

COLUNAS_GOLD = ["sigla_uf", "codigo_cbo_subgrupo", "competencia_prevista", "horizonte_h",
                "previsao_admissoes", "previsao_bruta", "modelo", "versao_modelo",
                "versao_features", "ultimo_mes_real", "versao_previsao", "dt_processamento"]
df_prev_spark = spark.createDataFrame(df_prev[COLUNAS_GOLD].astype({c: str for c in KEYS}))

ja_existe = 0
try:
    ja_existe = spark.table(TABELA_PREVISOES).where(F.col("versao_previsao") == VERSAO_PREVISAO).count()
except Exception:
    ja_existe = 0  # tabela ainda não existe

if ja_existe > 0:
    print(f"Versão {VERSAO_PREVISAO} já gravada ({ja_existe} linhas) — append ignorado (idempotente).")
else:
    df_prev_spark.write.mode("append").saveAsTable(TABELA_PREVISOES)
    print(f"Gravado: {df_prev_spark.count()} linhas em {TABELA_PREVISOES} (v{VERSAO_PREVISAO}).")
print(f"Previsões brutas negativas clipadas em 0: {negativos} de {len(df_prev)} ({negativos / len(df_prev):.1%})")

# Validação de leitura de volta (fonte da verdade = a tabela gold, não o DataFrame em memória)
chk = spark.table(TABELA_PREVISOES).where(F.col("versao_previsao") == VERSAO_PREVISAO).agg(
    F.count("*").alias("linhas"),
    F.sum(F.when(F.col("previsao_admissoes").isNull(), 1).otherwise(0)).alias("nulos"),
    F.countDistinct("sigla_uf", "codigo_cbo_subgrupo").alias("series"),
    F.min("competencia_prevista").alias("primeiro_mes"),
    F.max("competencia_prevista").alias("ultimo_mes_previsto"),
).first()
print(f"Gold lida de volta: {chk['linhas']} linhas | {chk['series']} séries | nulos: {chk['nulos']} | "
      f"{chk['primeiro_mes']} a {chk['ultimo_mes_previsto']}")
assert chk["linhas"] == qtd_series * HORIZONTE and chk["nulos"] == 0, "Validação da gold falhou"
print("Top-10 séries por previsão h=1 (sanidade):")
display(
    spark.table(TABELA_PREVISOES)
    .where((F.col("versao_previsao") == VERSAO_PREVISAO) & (F.col("horizonte_h") == 1))
    .orderBy(F.desc("previsao_admissoes"))
    .limit(10)
)

## Método, limitações e consumo

* **Recursivo h=1** — cada mês do horizonte é previsto como h=1 (horizonte validado no backtest do 036: MASE 0,80 na amostra top-20) e a previsão de admissões realimenta o histórico. Desligamentos e salário de meses futuros são desconhecidos → features correspondentes nulas a partir do 2º horizonte (o LightGBM trata NaN nativamente; degrade documentado).
* **h=1 totalmente fundamentado** — o primeiro mês previsto usa apenas lags/médias/salário reais até o último mês da wide table (2026-08 na v0.1.0); a partir dele, a previsão depende progressivamente de previsões.
* **Histórico substituído** — o artefato embute o histórico do treino (até 2026-01); este notebook o substitui pelos últimos 13 meses da wide table vigente, aproveitando todo o dado real disponível. O modelo em si (regressor LightGBM) permanece EXATAMENTE o registrado.
* **Clip de negativos** — admissões não podem ser negativas; a bruta é preservada em `previsao_bruta` e a oficial (`previsao_admissoes`) é clipada em 0, com contagem reportada.
* **Idempotência** — append por `versao_previsao`; re-executar com a mesma versão não duplica. Para um novo lote (após atualizar a wide table ou o modelo), bumpar `versao_previsao`.
* **Consumo** — dashboards/BI leem `workspace.gold.previsoes_admissoes` filtrando `versao_previsao` vigente; consumo on-demand externo usa o endpoint `demanda_admissoes_lgbm`.
* **Evoluções** — intervalo de previsão (quantiles via LightGBM), retreino mensal automatizado (pipeline 036→037), preenchimento de desl/sal futuros com modelo auxiliar.